<a href="https://colab.research.google.com/github/meriame-elhasany/sme_app/blob/main/Another_copy_of_Yet_another_copy_of_Yet_another_copy_of_colab_debian_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Transformer Google Colab en Serveur Debian / Ubuntu Complet
### 🌐 Serveur Web + 🔑 Accès SSH + 🖥️ Bureau Graphique à Distance RDP (Windows)

Ce notebook configure l'environnement virtuel de Google Colab en un **véritable serveur Linux (Ubuntu/Debian)** avec :
- 🛡️ **Accès Root complet (`sudo`)**
- 🔑 **Serveur OpenSSH (`openssh-server`)** pour administrer votre serveur en ligne de commande
- 🖥️ **Serveur Bureau à Distance RDP (`xrdp` + XFCE4)** pour vous connecter avec l'application native Windows `mstsc`
- 🌐 **Serveur Web Nginx** (ou Node.js / Python FastAPI / Flask / PHP)
- 🔗 **Accès Public HTTPS instantané et gratuit** via **Cloudflare Tunnel** (aucun compte requis)
- 💻 **Tunnels d'accès SSH & RDP** via **Pinggy (sans compte)**, **Ngrok TCP** ou **tmate**
- 💾 **Persistance des données** via Google Drive

> ⚠️ **Rappel important :**
> - Les sessions Colab gratuites durent jusqu'à 12h max et s'interrompent en cas d'inactivité prolongée.
> - Vos fichiers et codes peuvent être sauvegardés de façon permanente sur Google Drive (`/content/drive`).

## 1️⃣ Vérification de l'environnement Système (Debian / Ubuntu)

In [ ]:
# Affichage de la version Linux et des ressources système
!cat /etc/os-release
!uname -a
!echo "=== RAM ===" && free -h
!echo "=== CPU ===" && lscpu | grep "Model name\|CPU(s):"
!lsblk

PRETTY_NAME="Ubuntu 24.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="24.04"
VERSION="24.04.5 LTS (Noble Numbat)"
VERSION_CODENAME=noble
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=noble
LOGO=ubuntu-logo
Linux db191823f69f 6.6.159+ #1 SMP Thu Oct 22 03:07:54 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
=== RAM ===
               total        used        free      shared  buff/cache   available
Mem:            12Gi       897Mi       8.8Gi       2.2Mi       3.2Gi        11Gi
Swap:             0B          0B          0B
=== CPU ===
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
NUMA node0 CPU(s):                       0,1
NAME    MAJ:MIN RM   SIZE RO TYPE MOUNTPOINTS
loop0     7:0    0   110G  0 loop 
sda       8:0    0    23G  1 disk

## 2️⃣ (Optionnel) Montage de Google Drive pour la persistance
Exécutez cette cellule si vous souhaitez stocker vos applications web, données ou configurations de manière permanente.

In [ ]:
from google.colab import drive
import os

# Montage de Google Drive
drive.mount('/content/drive')

# Création d'un dossier pour vos projets s'il n'existe pas déjà
app_storage = '/content/drive/MyDrive/colab_server_apps'
os.makedirs(app_storage, exist_ok=True)
print(f"✅ Dossier de persistance prêt : {app_storage}")

Mounted at /content/drive
✅ Dossier de persistance prêt : /content/drive/MyDrive/colab_server_apps


In [ ]:
%%bash
sudo apt-get update -qq

sudo apt-get install -y -qq  openssh-server

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## 3️⃣ Mise à jour du système & Installation des outils de base
Installation de Nginx, OpenSSH, Git, Curl, Htop, Net-tools et des dépendances nécessaires.

In [ ]:
%%bash
echo "[+] Mise à jour du gestionnaire de paquets APT..."
sudo apt-get update -qq

echo "[+] Installation des outils de base, OpenSSH et Nginx..."
sudo apt-get install -y -qq nginx openssh-server curl wget git unzip htop net-tools tmux jq

echo "[+] Démarrage du serveur Nginx..."
sudo service nginx restart
sudo service nginx status

[+] Mise à jour du gestionnaire de paquets APT...
[+] Installation des outils de base, OpenSSH et Nginx...
Selecting previously unselected package libnl-genl-3-200:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../libnl-genl-3-200_3.7.0-0.3build1.1_amd64.deb ...
Unpacking libnl-genl-3-200:amd64 (3.7.0-0.3build1.1) ...
Selecting previously unselected package htop.
Preparing to unpack .../htop_3.3.0-4build1_amd64.deb ...
Unpacking htop (3.3.0-4build1) ...
Selecting previously unselected package nginx-common.
Preparing to unpack .../nginx-common_1.24.0-2ubuntu7.19_all.deb ...
Unpacking nginx-common (1.24.0-2ubuntu7.19) ...
Selecting previously unselected package nginx.
Preparing to unpack .../nginx_1.24.0-2ubuntu7.19_amd64.deb ...
Unpacking nginx (1.24.0-2ubuntu7.19) ...
Setting up nginx-common (1.24.0-2ubuntu7.19) ...
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog bas

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 4.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


## 4️⃣ Configuration du Serveur OpenSSH (`openssh-server`)
Cette étape configure le démon SSH standard de Debian/Ubuntu :
- Définition du mot de passe pour les connexions SSH et RDP
- Activation de `PermitRootLogin` et `PasswordAuthentication` dans `/etc/ssh/sshd_config`
- Démarrage du service SSH sur le port 22

In [ ]:
# ===========================================================
# Choisissez le mot de passe pour vos connexions SSH et RDP :
# ===========================================================
SERVER_PASSWORD = "debian123"  # <-- Modifiez ce mot de passe si souhaité

import os

# 1. Définition du mot de passe root et création d'un utilisateur standard 'colabuser'
os.system(f'echo "root:{SERVER_PASSWORD}" | chpasswd')
os.system(f'useradd -m -s /bin/bash -G sudo colabuser 2>/dev/null || true')
os.system(f'echo "colabuser:{SERVER_PASSWORD}" | chpasswd')

# 2. Configuration d'OpenSSH pour autoriser la connexion root par mot de passe
sshd_config = """
Port 22
PermitRootLogin yes
PasswordAuthentication yes
ChallengeResponseAuthentication no
UsePAM yes
X11Forwarding yes
PrintMotd no
AcceptEnv LANG LC_*
Subsystem sftp /usr/lib/openssh/sftp-server
"""

with open("/etc/ssh/sshd_config", "w") as f:
    f.write(sshd_config.strip())

# 3. Création du dossier /run/sshd requis par Debian/Ubuntu
os.makedirs("/run/sshd", exist_ok=True)

# 4. Redémarrage du service SSH
!service ssh restart

print("="*65)
print("✅ Serveur OpenSSH configuré et actif sur le port 22 !")
print(f"👤 Utilisateurs disponibles : 'root' ou 'colabuser'")
print(f"🔑 Mot de passe : {SERVER_PASSWORD}")
print("="*65)

[ OK ]
✅ Serveur OpenSSH configuré et actif sur le port 22 !
👤 Utilisateurs disponibles : 'root' ou 'colabuser'
🔑 Mot de passe : debian123


In [ ]:
ssh -p 22 -o StrictHostKeyChecking=no -o ServerAliveInterval=30 -R0:localhost:22 tcp@iot.aiot.ma

SyntaxError: invalid syntax (2341813487.py, line 1)

## 5️⃣ Connexion SSH à Distance (En Ligne de Commande)
Choisissez l'une des méthodes ci-dessous pour vous connecter en SSH depuis votre terminal Windows (PowerShell, PuTTY ou VS Code) :

### 🟢 SSH Méthode A : Tunnel Pinggy (Recommandée : Sans compte, Zéro inscription)
Génère une commande SSH directe sans avoir besoin de créer de compte.

In [ ]:
import subprocess
import time
import re
import os

# Arrêter les tunnels pinggy SSH précédents
!pkill -f "pinggy.io.*22" 2>/dev/null || true

log_file = "/content/pinggy_ssh.log"
if os.path.exists(log_file):
    os.remove(log_file)

print("[+] Lancement du tunnel SSH avec Pinggy...")
cmd = f"ssh -p 443 -o StrictHostKeyChecking=no -o ServerAliveInterval=30 -R0:localhost:22 tcp@a.pinggy.io > {log_file} 2>&1 &"
subprocess.Popen(cmd, shell=True)

ssh_cmd = None
for _ in range(20):
    time.sleep(1)
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            content = f.read()
            match = re.search(r'ssh\s+-p\s+(\d+)\s+([a-zA-Z0-9.-]+)', content)
            if match:
                port = match.group(1)
                host = match.group(2)
                ssh_cmd = f"ssh root@{host} -p {port}"
                break

if ssh_cmd:
    print("\n" + "="*65)
    print("🎉 VOTRE ACCÈS SSH EST PRÊT !")
    print("="*65)
    print(f"Copiez et collez cette commande dans votre PowerShell Windows :\n")
    print(f"👉  {ssh_cmd}")
    print(f"\n(Mot de passe : celui configuré dans la cellule 4)")
    print("="*65)
else:
    print("Contenu du log Pinggy :")
    !cat /content/pinggy_ssh.log

^C
[+] Lancement du tunnel SSH avec Pinggy...
Contenu du log Pinggy :
Pseudo-terminal will not be allocated because stdin is not a terminal.
Allocated port 7 for remote forward to localhost:22
You are not authenticated.
Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io
tcp://emqzr-35-196-107-84.run.pinggy-free.link:37173


### 🔵 SSH Méthode B : Tunnel Ngrok TCP (Pour VS Code Remote SSH)
Idéal si vous avez un compte gratuit [ngrok.com](https://dashboard.ngrok.com).

In [ ]:
NGROK_AUTHTOKEN = ""  # <-- Collez votre token ngrok ici si vous en avez un

if not NGROK_AUTHTOKEN:
    print("ℹ️ Si vous avez un compte ngrok, renseignez la variable NGROK_AUTHTOKEN.")
    print("👉 Sinon, utilisez la Méthode A (Pinggy) ou C (tmate) qui fonctionnent sans aucun compte.")
else:
    import json, urllib.request, subprocess, time
    !pip install -q pyngrok
    !ngrok config add-authtoken {NGROK_AUTHTOKEN}
    !pkill -f "ngrok tcp 22" 2>/dev/null || true
    subprocess.Popen("ngrok tcp 22 > /dev/null 2>&1 &", shell=True)
    time.sleep(3)
    try:
        req = urllib.request.Request("http://localhost:4040/api/tunnels")
        with urllib.request.urlopen(req) as res:
            data = json.loads(res.read().decode('utf-8'))
            public_url = [t['public_url'] for t in data['tunnels'] if t['proto'] == 'tcp'][0]
            host, port = public_url.replace("tcp://", "").split(":")
            print("\n" + "="*65)
            print(f"🎉 TUNNEL NGROK SSH ACTIF : ssh root@{host} -p {port}")
            print("="*65)
    except Exception as e:
        print(f"❌ Erreur ngrok : {e}")

ℹ️ Si vous avez un compte ngrok, renseignez la variable NGROK_AUTHTOKEN.
👉 Sinon, utilisez la Méthode A (Pinggy) ou C (tmate) qui fonctionnent sans aucun compte.


## 6️⃣ 🖥️ Serveur Bureau à Distance RDP pour Windows (`xrdp` + XFCE4)

Cette section installe l'environnement de bureau graphique léger **XFCE4** et le serveur **XRDP** (port 3389) pour vous permettre de contrôler votre serveur Debian avec l'application native **Connexion Bureau à distance de Windows (`mstsc.exe`)**.

In [ ]:
%%bash
echo "[+] Installation de l'environnement graphique XFCE4 et du serveur XRDP..."
export DEBIAN_FRONTEND=noninteractive
sudo apt-get update -qq
sudo apt-get install -y -qq xfce4 xfce4-goodies xrdp dbus-x11 x11-xserver-utils

echo "[+] Configuration des sessions utilisateurs pour XRDP..."
# Configuration de la session XFCE pour root et colabuser
echo "startxfce4" | sudo tee /root/.xsession > /dev/null
sudo chmod +x /root/.xsession
echo "startxfce4" | sudo tee /home/colabuser/.xsession > /dev/null
sudo chown colabuser:colabuser /home/colabuser/.xsession
sudo chmod +x /home/colabuser/.xsession

# Autorisation du groupe ssl-cert pour xrdp
sudo adduser xrdp ssl-cert > /dev/null 2>&1 || true

# Configuration de la qualité et des performances XRDP
sudo sed -i 's/max_bpp=32/max_bpp=16/g' /etc/xrdp/xrdp.ini
sudo sed -i 's/xserverbpp=24/xserverbpp=16/g' /etc/xrdp/xrdp.ini

# Démarrage du service XRDP
sudo service xrdp restart
sudo service xrdp status
echo "✅ Serveur RDP (XRDP) actif sur le port local 3389 !"

[+] Installation de l'environnement graphique XFCE4 et du serveur XRDP...
(Reading database ... 122879 files and directories currently installed.)
Removing fuse (2.9.9-8.1build1) ...
Selecting previously unselected package liblocale-gettext-perl.
(Reading database ... 122869 files and directories currently installed.)
Preparing to unpack .../0-liblocale-gettext-perl_1.07-6ubuntu5_amd64.deb ...
Unpacking liblocale-gettext-perl (1.07-6ubuntu5) ...
Selecting previously unselected package keyboard-configuration.
Preparing to unpack .../1-keyboard-configuration_1.226ubuntu1.1_all.deb ...
Unpacking keyboard-configuration (1.226ubuntu1.1) ...
Preparing to unpack .../2-systemd-sysv_255.4-1ubuntu8.18_amd64.deb ...
Unpacking systemd-sysv (255.4-1ubuntu8.18) over (255.4-1ubuntu8.17) ...
Preparing to unpack .../3-systemd-dev_255.4-1ubuntu8.18_all.deb ...
Unpacking systemd-dev (255.4-1ubuntu8.18) over (255.4-1ubuntu8.17) ...
Preparing to unpack .../4-libpam-systemd_255.4-1ubuntu8.18_amd64.deb ...
U

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 621.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


### 🔗 Exposer le Serveur RDP vers votre PC Windows
Exécutez cette cellule pour créer le tunnel d'accès à distance RDP. Elle vous fournira l'adresse exacte à copier dans **Connexion Bureau à distance (`mstsc`)**.

In [ ]:
import subprocess
import time
import re
import os

# Arrêter les tunnels RDP précédents
!pkill -f "pinggy.io.*3389" 2>/dev/null || true

log_file = "/content/pinggy_rdp.log"
if os.path.exists(log_file):
    os.remove(log_file)

print("[+] Création du tunnel RDP (port 3389) avec Pinggy (aucun compte requis)...")
cmd = f"ssh -p 443 -o StrictHostKeyChecking=no -o ServerAliveInterval=30 -R0:localhost:3389 tcp@a.pinggy.io > {log_file} 2>&1 &"
subprocess.Popen(cmd, shell=True)

rdp_host_port = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            content = f.read()
            # Recherche du host et port
            match = re.search(r'ssh\s+-p\s+(\d+)\s+([a-zA-Z0-9.-]+)', content)
            if match:
                port = match.group(1)
                host = match.group(2)
                rdp_host_port = f"{host}:{port}"
                break

if rdp_host_port:
    print("\n" + "="*70)
    print("🎉 VOTRE BUREAU À DISTANCE (RDP) WINDOWS EST PRÊT !")
    print("="*70)
    print("👉 INSTRUCTIONS DE CONNEXION DEPUIS WINDOWS :\n")
    print("1. Sur votre PC Windows, appuyez sur [Touche Windows + R]")
    print("2. Tapez 'mstsc' et appuyez sur Entrée (ou ouvrez 'Connexion Bureau à distance')")
    print(f"3. Dans le champ 'Ordinateur', collez exactement :\n\n    👉  {rdp_host_port}\n")
    print("4. Cliquez sur 'Connexion'. Ignorez l'avertissement de certificat s'il apparaît.")
    print(f"5. Sur l'écran XRDP, connectez-vous avec :")
    print(f"   - Session : Xorg")
    print(f"   - Utilisateur : root (ou colabuser)")
    print(f"   - Mot de passe : (celui défini dans la cellule 4)")
    print("="*70)
else:
    print("Erreur de création du tunnel RDP. Log :")
    !cat /content/pinggy_rdp.log

^C
[+] Création du tunnel RDP (port 3389) avec Pinggy (aucun compte requis)...
Erreur de création du tunnel RDP. Log :
Pseudo-terminal will not be allocated because stdin is not a terminal.
Allocated port 5 for remote forward to localhost:3389
You are not authenticated.
Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io
tcp://kjics-35-196-107-84.run.pinggy-free.link:36665


## 7️⃣ Déploiement d'une Application Web de Démonstration
Créons une page web d'accueil dans `/var/www/html` pour vérifier que le serveur HTTP répond correctement.

In [ ]:
%%bash
cat << 'EOF' | sudo tee /var/www/html/index.html > /dev/null
<!DOCTYPE html>
<html lang="fr">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Serveur Debian sur Google Colab</title>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap" rel="stylesheet">
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; font-family: 'Inter', sans-serif; }
        body { background: #0f172a; color: #f8fafc; display: flex; align-items: center; justify-content: center; min-height: 100vh; padding: 20px; }
        .card { background: #1e293b; border: 1px solid #334155; border-radius: 16px; padding: 40px; max-width: 680px; width: 100%; box-shadow: 0 20px 25px -5px rgba(0, 0, 0, 0.5); text-align: center; }
        .badge { background: #10b981; color: #042f2e; padding: 6px 14px; border-radius: 9999px; font-weight: 700; font-size: 0.85rem; display: inline-block; margin-bottom: 20px; text-transform: uppercase; letter-spacing: 0.05em; }
        h1 { font-size: 2.2rem; margin-bottom: 12px; color: #38bdf8; }
        p { color: #94a3b8; font-size: 1.05rem; line-height: 1.6; margin-bottom: 25px; }
        .stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 15px; margin-bottom: 30px; text-align: left; }
        .stat-item { background: #0f172a; padding: 15px; border-radius: 10px; border: 1px solid #334155; }
        .stat-label { font-size: 0.75rem; color: #64748b; text-transform: uppercase; font-weight: 600; }
        .stat-value { font-size: 1.05rem; font-weight: 700; color: #f1f5f9; margin-top: 5px; }
        .btn { display: inline-block; background: #2563eb; color: white; padding: 12px 28px; border-radius: 8px; text-decoration: none; font-weight: 600; transition: background 0.2s; }
        .btn:hover { background: #1d4ed8; }
    </style>
</head>
<body>
    <div class="card">
        <span class="badge">● En ligne & Opérationnel</span>
        <h1>Serveur Web Colab (Debian/Ubuntu)</h1>
        <p>Votre conteneur Google Colab est configuré en tant que serveur Linux complet avec accès SSH, Bureau à distance RDP et Nginx.</p>
        <div class="stats">
            <div class="stat-item">
                <div class="stat-label">Accès SSH</div>
                <div class="stat-value">Port 22</div>
            </div>
            <div class="stat-item">
                <div class="stat-label">Bureau RDP</div>
                <div class="stat-value">Port 3389 (XFCE)</div>
            </div>
            <div class="stat-item">
                <div class="stat-label">Serveur Web</div>
                <div class="stat-value">Port 80 (Nginx)</div>
            </div>
        </div>
        <a href="#" class="btn">Serveur Prêt pour vos Applications</a>
    </div>
</body>
</html>
EOF

sudo service nginx reload
echo "✅ Page de démonstration déployée sur Nginx !"

 * Reloading nginx configuration nginx
   ...done.
✅ Page de démonstration déployée sur Nginx !


## 8️⃣ Exposer le Serveur Web sur Internet avec Cloudflare Tunnel
Nous utilisons **Cloudflare Tunnel (`cloudflared`)** pour exposer le serveur web Nginx (port 80) :
- Ne nécessite **aucun compte ni inscription**
- Fournit un lien public sécurisé `https://xxxxxx.trycloudflare.com`
- Offre une connexion HTTPS rapide et illimitée

In [ ]:
import subprocess
import time
import re
import os

# 1. Télécharger cloudflared si absent
if not os.path.exists('/usr/local/bin/cloudflared'):
    print("[+] Téléchargement de Cloudflare Tunnel...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !sudo dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    !rm -f cloudflared-linux-amd64.deb
    print("✅ Cloudflared installé.")

# 2. Arrêter les tunnels web existants
!pkill -f "cloudflared tunnel" 2>/dev/null || true

# 3. Lancer le tunnel vers le port 80 de Nginx (ou votre port d'application)
PORT = 80
print(f"[+] Démarrage du tunnel vers http://localhost:{PORT}...")
log_file = "/content/cloudflared.log"
cmd = f"cloudflared tunnel --url http://localhost:{PORT} --logfile {log_file} > /dev/null 2>&1 &"
subprocess.Popen(cmd, shell=True)

# 4. Extraire l'URL publique
url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists(log_file):
        with open(log_file, 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                url = match.group(0)
                break

if url:
    print("\n" + "="*60)
    print(f"🎉 VOTRE APPLICATION WEB EST ACCESSIBLE EN LIGNE SUR :")
    print(f"👉 {url}")
    print("="*60 + "\n")
else:
    print("❌ Impossible de récupérer l'URL. Consultez le log :")
    !cat /content/cloudflared.log

[+] Téléchargement de Cloudflare Tunnel...
✅ Cloudflared installé.
^C
[+] Démarrage du tunnel vers http://localhost:80...

🎉 VOTRE APPLICATION WEB EST ACCESSIBLE EN LIGNE SUR :
👉 https://somewhere-requested-threatening-morgan.trycloudflare.com



## 9️⃣ Exemples de Déploiement d'Applications Web

### A. Application Python (FastAPI / Uvicorn ou Flask)
Voici un modèle pour exécuter une API Python FastAPI ou Flask en arrière-plan :

In [ ]:
# Installation de FastAPI et Uvicorn
!pip install -q fastapi uvicorn

# Création d'une application FastAPI d'exemple
api_code = '''
from fastapi import FastAPI
from fastapi.responses import HTMLResponse

app = FastAPI(title="Colab Debian Web Server")

@app.get("/", response_class=HTMLResponse)
def read_root():
    return "<h1>Bonjour depuis votre API FastAPI sur Debian Colab!</h1><p>Documentation interactive : <a href='/docs'>/docs</a></p>"

@app.get("/api/status")
def get_status():
    return {"server": "Debian / Colab", "status": "running", "port": 8000}
'''

with open("/content/app.py", "w") as f:
    f.write(api_code)

print("✅ Fichier /content/app.py créé !")
print("Pour le lancer en arrière-plan : !nohup uvicorn app:app --host 0.0.0.0 --port 8000 > /content/api.log 2>&1 &")

### B. Application Node.js / Express
Pour installer Node.js et lancer une application Express :

In [ ]:
%%bash
# Installation de Node.js 20 LTS si souhaité
if ! command -v node &> /dev/null; then
    curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
    sudo apt-get install -y -qq nodejs
fi
node -v
npm -v

### C. Application PHP / LAMP (Apache & MySQL / SQLite)
Pour exécuter des applications PHP / Laravel / WordPress :

In [ ]:
%%bash
# Installation de PHP et des extensions habituelles
sudo apt-get install -y -qq php-fpm php-cli php-mysql php-sqlite3 php-curl php-xml php-mbstring
php -v

## 🔟 Anti-Déconnexion / Keep-Alive (Maintien de Session Colab)
Google Colab coupe les sessions inactives au bout d'un certain temps.

### Méthode 1 : Script JavaScript pour le navigateur
Appuyez sur `F12` (ou `Ctrl + Shift + I`) dans votre navigateur, allez dans l'onglet **Console**, collez ce code et appuyez sur **Entrée** :

```javascript
function ClickConnect(){
    console.log("Working... Keep-alive active");
    document.querySelector("colab-connect-button").click();
}
setInterval(ClickConnect, 60000);
```

### Méthode 2 : Boucle Python de maintien en vie
Exécutez la cellule suivante pour laisser le notebook actif pendant que votre serveur web, SSH et RDP tournent en arrière-plan.

In [ ]:
import time
import datetime

print("🟢 Serveur Debian, SSH, RDP & Nginx actifs sur Google Colab. Maintien en vie lancé...")
print("Appuyez sur 'Stop' (bouton d'interruption de cellule) pour arrêter le keep-alive.\n")

try:
    while True:
        now = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"[{now}] Serveur Debian opérationnel...", end="\r")
        time.sleep(30)
except KeyboardInterrupt:
    print("\n🛑 Keep-alive interrompu par l'utilisateur.")